In [73]:
   import pandas as pd
   pd.__version__

'3.0.6'

In [74]:
from pathlib import Path
DATA = Path("..") / "data" / "processed"

daily   = pd.read_csv(DATA / "daily_master.csv", dtype={"user_id": str}, parse_dates=["date"])
wear    = pd.read_csv(DATA / "wear_log.csv",     dtype={"user_id": str}, parse_dates=["date"])
profile = pd.read_csv(DATA / "user_profile.csv", dtype={"user_id": str}, parse_dates=["first_date", "last_date"])

daily.shape, wear.shape, profile.shape

((757, 41), (1023, 6), (33, 59))

   ## Layer 1 · Data overview

In [75]:
   print("Users:", daily["user_id"].nunique())
   print("Clean days:", len(daily))
   wear["wear_status"].value_counts()

Users: 33
Clean days: 757


wear_status
worn        757
stopped      83
not_worn     77
partial      73
last_day     33
Name: count, dtype: int64

In [76]:
in_study = wear[wear["wear_status"] != "stopped"]
wear_rate = (in_study["wear_status"] == "worn").mean()

print("Days in study:", len(in_study))
print(f"Wear rate: {wear_rate:.1%}")

Days in study: 940
Wear rate: 80.5%


In [77]:
in_study = wear[~wear["wear_status"].isin(["stopped", "last_day"])]

In [78]:
days_per_user = daily.groupby("user_id").size().sort_values()
days_per_user.describe()

count    33.000000
mean     22.939394
std       8.291334
min       2.000000
25%      19.000000
50%      27.000000
75%      29.000000
max      30.000000
dtype: float64

In [79]:
days_per_user.head(6)

user_id
4057192912     2
6775888955     3
1927972279     7
4020332650     9
1844505072    12
8253242879    15
dtype: int64

In [80]:
import plotly.express as px

chart_df = days_per_user.reset_index(name="clean_days")

fig = px.bar(chart_df, x="user_id", y="clean_days",
             title="Clean days per person (max 30)",
             labels={"user_id": "Person", "clean_days": "Clean days"})
fig.add_hline(y=7, line_dash="dash", annotation_text="7-day rule")
fig.show()

In [81]:
cols = ["tracks_sleep", "has_hr_data", "logs_weight", "logs_workouts"]
coverage = profile[cols].sum().to_frame("users")
coverage["share"] = (coverage["users"] / len(profile)).round(2)
coverage

,users,share
tracks_sleep,24,0.73
has_hr_data,14,0.42
logs_weight,8,0.24
logs_workouts,3,0.09


In [82]:
profile["features_used"].value_counts().sort_index()

features_used
0     7
1    18
2     7
3     1
Name: count, dtype: int64

### Layer 1 · Key findings
- **33 people, 757 clean days** (12 Apr – 11 May 2016).
- **Wear rate 83.5%** of days in study (each person's last day excluded — export cut-off).
- More days were lost to **drop-outs (83)** than to not wearing (77) or partial wear (73) → retention matters.
- Typical person: **27 clean days** (median); average 22.9 is pulled down by a patchy tail. 2 people are below the 7-day rule.
- Two groups: a large loyal core (25+ days) and a smaller patchy tail.
- Feature use drops as effort rises: **sleep 73% → weight 24% → workouts 9%**. Heart rate (42%) depends on the device model.
- **7 people use no extra feature**; sleep is the gateway feature (16 of the 18 single-feature users).
- Sample sizes for later: sleep ≤ 24, heart rate 13–14, weight 8 → always show n.

   ## Layer 2 · One measure at a time
   ### Daily steps

In [83]:
   daily["steps"].describe().round(0)

count      757.0
mean      9065.0
std       4481.0
min        678.0
25%       5832.0
50%       8758.0
75%      11550.0
max      36019.0
Name: steps, dtype: float64

In [84]:
fig = px.histogram(daily, x="steps",
                   title="Most days fall short of 10k, but many stop just past it (757 days)",
                   labels={"steps": "Steps per day"})
fig.update_traces(xbins=dict(start=0, end=37000, size=1000),
                  hovertemplate="%{x} steps<br>%{y} days<extra></extra>")
fig.add_vline(x=daily["steps"].median(), line_dash="dash",
              annotation_text="Median 8,758", annotation_position="top left")
fig.add_vline(x=10000, line_dash="dot",
              annotation_text="10k goal", annotation_position="top right")
fig.update_yaxes(title_text="Days")
fig.update_layout(bargap=0.05)
fig.show()

print(f"Days hitting 10k: {daily['goal_10k_met'].mean():.0%}")

Days hitting 10k: 40%


### Active minutes

In [85]:
daily[["active_min", "mvpa_min", "sedentary_awake_min"]].describe().round(0)

,active_min,mvpa_min,sedentary_awake_min
count,750.0,750.0,365.0
mean,272.0,42.0,702.0
std,86.0,45.0,111.0
min,55.0,0.0,413.0
25%,216.0,0.0,634.0
50%,271.0,30.0,713.0
75%,331.0,65.0,767.0
max,552.0,275.0,1058.0


In [86]:
mvpa = daily["mvpa_min"].dropna()

fig = px.histogram(mvpa, x="mvpa_min",
                   title="1 in 3 days has no real exercise at all; on the rest, effort varies widely",
                   labels={"mvpa_min": "Moderate-to-vigorous minutes per day"})
fig.update_traces(xbins=dict(start=0, end=280, size=10),
                  hovertemplate="%{x} min<br>%{y} days<extra></extra>")
fig.add_vline(x=mvpa.median(), line_dash="dash",
              annotation_text=f"Median {mvpa.median():.0f} min", annotation_position="top right")
fig.update_yaxes(title_text="Days")
fig.update_layout(bargap=0.05)
fig.show()

print(f"Days with zero MVPA: {mvpa.eq(0).mean():.0%}")

Days with zero MVPA: 31%


In [87]:
sleep = pd.read_csv(DATA / "sleep_nights.csv", dtype={"user_id": str}, parse_dates=["date"])
nights = sleep[sleep["reliable_night"]]

print("All nights:", len(sleep), "· Reliable nights:", len(nights),
      "· People:", nights["user_id"].nunique())
nights["night_hours_asleep"].describe().round(2)

All nights: 410 · Reliable nights: 384 · People: 19


count    384.00
mean       7.12
std        1.49
min        3.27
25%        6.12
50%        7.22
75%        8.10
max       11.67
Name: night_hours_asleep, dtype: float64

In [88]:
nights["sleep_category"].value_counts(normalize=True).round(3)

sleep_category
7–9 h    0.492
< 6 h    0.224
6–7 h    0.203
> 9 h    0.081
Name: proportion, dtype: float64

In [89]:
fig = px.histogram(nights, x="night_hours_asleep",
                      title="Only Half of Nights Reach 7-9 h; Short Nights Outnumber Long Ones"
      "<br><sup>384 Reliable Nights · 19 People</sup>",
                   labels={"night_hours_asleep": "Hours asleep"})
fig.update_traces(xbins=dict(start=3, end=12, size=0.5),
                  hovertemplate="%{x} h<br>%{y} nights<extra></extra>")
fig.add_vrect(x0=7, x1=9, fillcolor="green", opacity=0.12, line_width=0,
              annotation_text="7-9 h band", annotation_position="top left")
fig.update_yaxes(title_text="Nights", range=[0, 70])
fig.update_layout(bargap=0.05)
fig.show()

In [90]:
fig = px.strip(people, x="typical_sleep_hours",
               hover_data=["user_id", "reliable_nights"],
               title="12 of 16 People Typically Sleep 7-9 h; 4 Fall Short, None Oversleep"
      "<br><sup>One dot per person · median of their reliable nights (7+ nights each)</sup>",
               labels={"typical_sleep_hours": "Typical hours asleep (median)"})
fig.add_vrect(x0=7, x1=9, fillcolor="green", opacity=0.12, line_width=0)
fig.update_traces(marker_size=12)
fig.update_xaxes(range=[4.5, 9.5])
fig.update_layout(height=320)
fig.show()

### Sleep efficiency

In [91]:
nights["sleep_efficiency"].describe().round(3)

count    384.000
mean       0.920
std        0.083
min        0.535
25%        0.914
50%        0.943
75%        0.961
max        0.998
Name: sleep_efficiency, dtype: float64

In [92]:
low = nights[nights["sleep_efficiency"] < 0.85]
print("Nights below 85%:", len(low))
low["user_id"].value_counts()

Nights below 85%: 28


user_id
3977333714    27
8378563200     1
Name: count, dtype: int64

### Bedtime

In [93]:
bed = nights["bedtime_hrs_after_6pm"]
bed.describe().round(2)

count    384.00
mean       5.66
std        2.08
min        2.15
25%        4.28
50%        5.21
75%        6.79
max       18.42
Name: bedtime_hrs_after_6pm, dtype: float64

In [94]:
def to_clock(hours_after_6pm):
    minutes = round((18 + hours_after_6pm) * 60) % (24 * 60)
    h, m = divmod(minutes, 60)
    return f"{h % 12 or 12}:{m:02d} {'AM' if h < 12 else 'PM'}"

bed.quantile([0.25, 0.5, 0.75]).apply(to_clock)

0.25    10:17 PM
0.50    11:13 PM
0.75    12:47 AM
Name: bedtime_hrs_after_6pm, dtype: str

In [95]:
wake = pd.to_timedelta(nights["wake_time"] + ":00").dt.total_seconds() / 3600
wake.quantile([0.25, 0.5, 0.75]).apply(lambda h: to_clock(h - 18))

0.25    6:07 AM
0.50    7:04 AM
0.75    8:32 AM
Name: wake_time, dtype: str

### Body metrics (BMI)

In [96]:
body = pd.read_csv(DATA / "user_body_profile.csv", dtype={"user_id": str})
body[["user_id", "bmi", "bmi_category", "n_weight_logs", "weight_habit"]].sort_values("bmi")

,user_id,bmi,bmi_category,n_weight_logs,weight_habit
2,2873212765,21.6,Normal,2,False
0,1503960366,22.6,Normal,2,False
6,6962181067,24.0,Normal,30,True
7,8877689391,25.5,Overweight,24,True
4,4558609924,27.2,Overweight,5,False
3,4319703577,27.4,Overweight,2,False
5,5577150313,28.0,Overweight,1,False
1,1927972279,47.5,Obese,1,False


### Heart rate

In [97]:
hr_cols = ["user_id", "hr_days", "typical_resting_hr",
           "typical_hr_minutes_above_100", "low_hr_data_user"]
hr_people = profile.loc[profile["has_hr_data"], hr_cols]
hr_people.sort_values("typical_resting_hr")

,user_id,hr_days,typical_resting_hr,typical_hr_minutes_above_100,low_hr_data_user
20,5577150313,26,48.5,109.0,False
15,4388161847,28,51.0,21.5,False
31,8792009665,16,55.0,22.5,False
19,5553957443,28,56.0,14.5,False
32,8877689391,29,57.0,80.0,False
12,4020332650,8,59.0,53.0,False
5,2022484408,25,60.0,70.0,False
8,2347167796,17,60.0,46.0,False
24,6962181067,30,61.0,65.5,False
21,6117666160,20,62.0,76.0,False


### Layer 2 · Key findings
- **Steps:** typical day **8,758** (median); middle half 5.8k–11.5k; long tail to 36k. **40% of days hit 10k.**
- **Goal effect:** ~50% more days land just past 10k (10,000–10,499) than just before it → people push to reach the goal.
- **Effort:** typical day **30 MVPA min**, but **31% of days have zero** — "no-exercise days" vs "exercise days". ~9 in 10 active minutes are light.
- **Sitting:** ~**12 h** a day sitting while awake (365 sleep-tracked days).
- **Sleep length:** typical night **7 h 13 min**; only **49%** of nights in 7–9 h, **43% under 7 h**, 8% over 9 h → the issue is *too little*, not too much. Per person: 12 of 16 in 7–9 h, 4 short, none long.
- **Sleep quality:** typical efficiency **94%** (~26 min awake in bed). **27 of 28 restless nights belong to one person** → not a group issue.
- **Timing:** bedtime **11:13 PM** (middle half 10:17 PM – 12:47 AM); wake **7:04 AM** (6:07–8:32 AM).
- **BMI (n = 8, self-selected):** 3 normal, 4 overweight, 1 obese; 5 of 8 logged only once or twice → snapshot only, no weight trends.
- **Heart rate (n = 12):** typical resting 55–63; elevated minutes vary 15–214 a day. Used as effort detail only (R7).

## Layer 3 · Patterns over time
### Hour of the day

In [98]:
hourly = pd.read_csv(DATA / "hourly_clean.csv", dtype={"user_id": str},
                     parse_dates=["date", "datetime"])
print("Rows:", len(hourly), "· Person-days:", hourly.groupby(["user_id", "date"]).ngroups)

by_hour = hourly.groupby("hour")["steps"].mean().round(0)
print("Sum of the 24 hourly averages:", by_hour.sum())
by_hour

Rows: 17987 · Person-days: 750
Sum of the 24 hourly averages: 9069.0


hour
0      46.0
1      26.0
2      18.0
3       6.0
4      14.0
5      50.0
6     209.0
7     355.0
8     501.0
9     515.0
10    561.0
11    528.0
12    644.0
13    634.0
14    634.0
15    479.0
16    586.0
17    658.0
18    707.0
19    688.0
20    420.0
21    363.0
22    282.0
23    145.0
Name: steps, dtype: float64

In [99]:
fig = px.bar(by_hour.reset_index(), x="hour", y="steps",
             title="Activity Builds Through the Day and Peaks at 6 PM After Work"
                   "<br><sup>Average steps per hour · 750 person-days · 33 people</sup>",
             labels={"hour": "Hour of day (0 = midnight)", "steps": "Average steps"})
fig.update_traces(hovertemplate="%{x}:00<br>%{y:,.0f} steps<extra></extra>")
fig.update_xaxes(dtick=1)
fig.show()

*Note: hourly patterns use the **average**, not the median — most hours are near zero, so medians hide the shape. The 24 averages add up to 9,069 ≈ daily average 9,065 ✅*

In [100]:
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
heat = (hourly.pivot_table(index="day_of_week", columns="hour",
                           values="steps", aggfunc="mean")
              .reindex(order))

fig = px.imshow(heat, aspect="auto", color_continuous_scale="Oranges",
                labels={"x": "Hour of day", "y": "", "color": "Avg steps"},
                title="Weekdays Peak After Work; Weekends Start Late and Peak at Midday"
                      "<br><sup>Average steps per hour · 750 person-days · 33 people</sup>")
fig.update_traces(hovertemplate="%{y}, %{x}:00<br>%{z:,.0f} steps<extra></extra>")
fig.update_xaxes(dtick=1)
fig.show()

In [101]:
daily["day_type"] = daily["is_weekend"].map({False: "Weekday", True: "Weekend"})
print(daily.groupby("day_type")["steps"].agg(["size", "median"]))

fig = px.box(daily, x="day_type", y="steps", points="outliers",
             title="Weekend Days Are Slightly Less Active Than Weekdays"
                   "<br><sup>Steps per day · all 757 days · 33 people</sup>",
             labels={"day_type": "", "steps": "Steps per day"})
fig.update_yaxes(rangemode="tozero")
fig.show()

          size  median
day_type              
Weekday    565  8925.0
Weekend    192  8113.5


In [102]:
per_person = (daily.pivot_table(index="user_id", columns="day_type",
                                values="steps", aggfunc="median")
                   .dropna())
per_person["weekend_higher"] = per_person["Weekend"] > per_person["Weekday"]

print("People compared:", len(per_person))
print("Walk more on weekends:", per_person["weekend_higher"].sum())

People compared: 31
Walk more on weekends: 19


In [103]:
import plotly.graph_objects as go

pp = per_person.copy()
pp["diff"] = pp["Weekend"] - pp["Weekday"]
pp = pp.sort_values("diff").reset_index()
pp = pp.merge(profile[["user_id", "week_pattern"]], on="user_id", how="left")        # NEW
colors = {"Weekend-active": "#2a9d8f", "Weekday-active": "#e76f51", "Steady": "#b0b0b0"}   # NEW

fig = go.Figure()

# 1) one line per person, coloured by their week pattern
for _, r in pp.iterrows():
    fig.add_trace(go.Scatter(
        x=[r["Weekday"], r["Weekend"]], y=[r["user_id"], r["user_id"]],
        mode="lines", line=dict(color=colors[r["week_pattern"]], width=3),    # CHANGED
        showlegend=False, hoverinfo="skip"))

# 2) the two sets of dots
fig.add_trace(go.Scatter(x=pp["Weekday"], y=pp["user_id"], mode="markers", name="Weekday",
                         marker=dict(size=10, color="grey"),
                         hovertemplate="%{y}<br>Weekday: %{x:,.0f} steps<extra></extra>"))
fig.add_trace(go.Scatter(x=pp["Weekend"], y=pp["user_id"], mode="markers", name="Weekend",
                         marker=dict(size=10, color="#FC4C02"),
                         hovertemplate="%{y}<br>Weekend: %{x:,.0f} steps<extra></extra>"))

fig.update_layout(
    title="Most People Walk More at Weekends, but Weekday High-Steppers Drop Sharply"
          "<br><sup>Median steps per person · green = weekend-active (16) · red = weekday-active (11)"
          " · grey = steady, within ±10% (4)</sup>",                                  # CHANGED
    xaxis_title="Median steps per day", height=750)
fig.update_yaxes(type="category", title="")
fig.show()

### Weekly trend

In [104]:
d4 = daily[daily["week_no"] <= 4]

days_per_week = d4.groupby(["user_id", "week_no"]).size().unstack(fill_value=0)
fixed_ids = days_per_week[(days_per_week >= 1).all(axis=1)].index
print("Fixed crowd:", len(fixed_ids), "people")

trend = pd.DataFrame({
    "users_all":   d4.groupby("week_no")["user_id"].nunique(),
    "steps_all":   d4.groupby("week_no")["steps"].median(),
    "steps_fixed": d4[d4["user_id"].isin(fixed_ids)].groupby("week_no")["steps"].median(),
})
trend

Fixed crowd: 27 people


,users_all,steps_all,steps_fixed
week_no,,,
1,33,8181.0,8163.0
2,31,8925.0,9169.5
3,30,8969.0,9232.0
4,28,8567.0,8663.0


In [105]:
t = trend.reset_index()
t["label"] = "Week " + t["week_no"].astype(str) + "<br>" + t["users_all"].astype(str) + " users"

fig = go.Figure()
fig.add_trace(go.Scatter(x=t["label"], y=t["steps_fixed"], mode="lines+markers",
                         name=f"Same {len(fixed_ids)} people (fixed crowd)",
                         line=dict(color="#FC4C02", width=3),
                         hovertemplate="Fixed crowd: %{y:,.0f} steps<extra></extra>"))
fig.add_trace(go.Scatter(x=t["label"], y=t["steps_all"], mode="lines+markers",
                         name="Everyone present that week",
                         line=dict(color="grey", dash="dash"),
                         hovertemplate="Everyone: %{y:,.0f} steps<extra></extra>"))
fig.update_layout(
    title="Activity Holds Steady All Month for Those Who Keep Wearing the Device"
          "<br><sup>Median steps per day · weeks 1–4 (week 5 has only 2 days) · users present under each week</sup>",
    yaxis_title="Median steps per day")
fig.update_yaxes(rangemode="tozero")
fig.show()

In [106]:
order = ["Morning", "Afternoon", "Evening", "Late", "No workouts"]
wo = (profile["preferred_workout_time"].value_counts()
            .reindex(order, fill_value=0)
            .rename_axis("time").reset_index(name="people"))

fig = px.bar(wo, x="time", y="people", text="people",
             title="Morning Is the Most Common Workout Time, but No Single Slot Dominates"
                   "<br><sup>Preferred workout time per person · 33 people · Morning 5–11 · Afternoon 12–4 PM"
                   " · Evening 5–9 PM · Late 10–11 PM</sup>",
             labels={"time": "", "people": "People"})
fig.update_traces(hovertemplate="%{x}: %{y} people<extra></extra>")
fig.show()
wo

,time,people
0,Morning,13
1,Afternoon,9
2,Evening,8
3,Late,0
4,No workouts,3


### Layer 3 · Key findings
- **Daily rhythm:** two humps — morning ramp 6–8 AM → lunchtime plateau (12–2 PM) → **3 PM dip** → **peak at 6 PM** (after work). Hourly averages add up to the daily average (9,069 ≈ 9,065) ✅
- **Weekday vs weekend rhythm:** weekdays peak **5–7 PM**; weekends start late (6–7 AM pale) and peak **10 AM–2 PM**. Busiest slot of the week: **Saturday 1 PM**. Weekend wake-up ~80 min later (6:46 → 8:05 AM).
- **Weekend paradox:** pooled weekend days are ~800 steps lower, yet **16 people are weekend-active vs 11 weekday-active** (4 steady). A few weekday high-steppers drop sharply on weekends and outweigh many small gains.
- **Weekend wear dip:** worn days 84.8% on weekdays vs 79.7% on weekends.
- **Monthly trend:** for the fixed crowd (27 people present all 4 weeks), activity **does not fade** (8.2k → 9.2k → 9.2k → 8.7k). The loss is **people who stop wearing**, not people moving less → retention first.
- **Workout time:** by people, **Morning 13 · Afternoon 9 · Evening 8 · Late 0 · No workouts 3**; by workout hours, evening leads (peak 7 PM). Morning slot is longer (7 h), so treat its lead with care.
- **Timing ideas:** weekday nudge ~5 PM, weekend ~10–11 AM, 3 PM "move" reminder, Friday-evening charge reminder, no late or early-weekend pushes.

## Layer 4 · Relationships & segments
### Activity segments (Q1)

In [107]:
seg_order  = ["Sedentary", "Low active", "Somewhat active", "Active"]
seg_colors = {"Sedentary": "#c6dbef", "Low active": "#6baed6",
              "Somewhat active": "#2171b5", "Active": "#08306b"}

seg = profile.dropna(subset=["activity_segment"])
print(seg["activity_segment"].value_counts().reindex(seg_order))

fig = px.strip(seg, x="typical_steps", color="activity_segment",
               category_orders={"activity_segment": seg_order},
               color_discrete_map=seg_colors, stripmode="overlay",
               hover_data=["user_id", "clean_days"],
               title="Two in Three People Walk 7,500+ Steps a Day; One in Three Fall Below"
                     "<br><sup>Typical daily steps per person (median) · 31 people with 7+ clean days"
                     " · dotted lines at 5k / 7.5k / 10k</sup>",
               labels={"typical_steps": "Typical daily steps", "activity_segment": "Segment"})
for x in [5000, 7500, 10000]:
    fig.add_vline(x=x, line_dash="dot", line_color="grey")
fig.update_traces(marker_size=12)
fig.update_layout(height=360)
fig.show()

activity_segment
Sedentary           5
Low active          5
Somewhat active    10
Active             11
Name: count, dtype: int64


In [108]:
print(pd.crosstab(seg["activity_segment"], seg["meets_activity_guideline"],
                  margins=True).reindex(seg_order + ["All"]))

seg.groupby("activity_segment")["weekly_mvpa_min"].median().reindex(seg_order)

meets_activity_guideline  False  True  All
activity_segment                          
Sedentary                     5     0    5
Low active                    5     0    5
Somewhat active               3     7   10
Active                        0    11   11
All                          13    18   31


activity_segment
Sedentary           59.0
Low active          46.0
Somewhat active    242.5
Active             419.0
Name: weekly_mvpa_min, dtype: float64

In [109]:
seg_profile = (seg.groupby("activity_segment")
                  .agg(people=("user_id", "size"),
                       steps=("typical_steps", "median"),
                       mvpa_min=("typical_mvpa_min", "median"),
                       distance_km=("typical_distance_km", "median"),
                       calories=("typical_calories", "median"),
                       goal_10k_rate=("goal_10k_rate", "median"),
                       clean_day_share=("clean_day_share", "median"),
                       features_used=("features_used", "mean"))
                  .reindex(seg_order)
                  .round(2))
seg_profile

,people,steps,mvpa_min,distance_km,calories,goal_10k_rate,clean_day_share,features_used
activity_segment,,,,,,,,
Sedentary,5,4595.00,0.0,3.08,2067.00,0.00,0.52,1.00
Low active,5,5862.00,0.0,4.20,1985.00,0.04,0.74,0.60
Somewhat active,10,8357.25,30.5,6.28,2634.75,0.25,0.87,1.20
Active,11,11658.00,58.0,7.89,2601.00,0.76,0.94,1.27


In [110]:
q1, q3 = seg["steps_cv"].quantile([0.25, 0.75])

fig = px.scatter(seg, x="typical_steps", y="steps_cv", color="activity_segment",
                 category_orders={"activity_segment": seg_order},
                 color_discrete_map=seg_colors,
                 hover_data=["user_id", "clean_days", "routine_consistency"],
                 title="Less Active People Tend to Have Less Regular Routines"
                       "<br><sup>One dot per person · higher = daily steps swing more · 31 people</sup>",
                 labels={"typical_steps": "Typical daily steps",
                         "steps_cv": "Day-to-day variation (CV)", "activity_segment": "Segment"})
fig.add_hline(y=q1, line_dash="dot", annotation_text="Very regular below")
fig.add_hline(y=q3, line_dash="dot", annotation_text="Irregular above")
fig.update_traces(marker_size=12)
fig.show()

print(seg.groupby("routine_consistency")["typical_steps"].median().round(0))

routine_consistency
Irregular       6648.0
Normal          8908.0
Very regular    9784.0
Name: typical_steps, dtype: float64


In [111]:
mins = (daily.dropna(subset=["active_min"])
             .groupby("user_id")[["lightly_active_min", "fairly_active_min", "very_active_min"]].mean()
             .join(seg.set_index("user_id")["activity_segment"], how="inner")
             .groupby("activity_segment").mean()
             .reindex(seg_order))
print(mins.round(0))

shares = mins.div(mins.sum(axis=1), axis=0).round(3)
shares

                  lightly_active_min  fairly_active_min  very_active_min
activity_segment                                                        
Sedentary                      178.0                4.0              4.0
Low active                     269.0                5.0              4.0
Somewhat active                236.0               18.0             24.0
Active                         229.0               22.0             39.0


,lightly_active_min,fairly_active_min,very_active_min
activity_segment,,,
Sedentary,0.958,0.020,0.022
Low active,0.969,0.017,0.014
Somewhat active,0.851,0.064,0.085
Active,0.789,0.077,0.134


In [114]:
long = shares.reset_index().melt(id_vars="activity_segment", var_name="effort", value_name="share")
long["effort"] = long["effort"].map({"lightly_active_min": "Light",
                                     "fairly_active_min": "Fairly active",
                                     "very_active_min": "Very active"})
long["label"] = long["share"].map("{:.0%}".format)

fig = px.bar(long, x="activity_segment", y="share", color="effort", text="label",
             category_orders={"activity_segment": seg_order,
                              "effort": ["Light", "Fairly active", "Very active"]},
             color_discrete_map={"Light": "#fdd0a2", "Fairly active": "#fd8d3c", "Very active": "#d94801"},
             title="Most Groups Move a Similar Amount of Time; What Differs Is How Hard"
                   "<br><sup>Share of active minutes by effort level · average per person · 31 people</sup>",
             labels={"activity_segment": "", "share": "Share of active minutes", "effort": "Effort"})
fig.update_yaxes(tickformat=".0%", range=[0, 1])
fig.update_traces(hovertemplate="%{x}<br>%{y:.0%}<extra></extra>")
totals = mins.sum(axis=1)
fig.update_xaxes(tickvals=seg_order,
                 ticktext=[f"{s}<br>{totals[s]:.0f} active min/day" for s in seg_order])
fig.update_layout(uniformtext_minsize=11, uniformtext_mode="hide")
fig.update_traces(hovertemplate="%{x}<br>%{fullData.name}: %{y:.0%}<extra></extra>")
fig.show()

In [115]:
km = (daily.groupby("user_id")[["light_active_km", "moderate_active_km", "very_active_km"]].mean()
           .join(seg.set_index("user_id")["activity_segment"], how="inner")
           .groupby("activity_segment").mean()
           .reindex(seg_order))
print(km.round(2))
km.div(km.sum(axis=1), axis=0).round(3)

                  light_active_km  moderate_active_km  very_active_km
activity_segment                                                     
Sedentary                    2.44                0.15            0.35
Low active                   4.02                0.16            0.26
Somewhat active              4.09                0.75            1.26
Active                       4.36                0.97            3.21


,light_active_km,moderate_active_km,very_active_km
activity_segment,,,
Sedentary,0.831,0.050,0.119
Low active,0.906,0.035,0.059
Somewhat active,0.671,0.123,0.206
Active,0.511,0.114,0.376


In [116]:
sl = hourly[hourly["stepless_effort"]]
print("Step-free effort hours:", len(sl), "· people:", sl["user_id"].nunique())
print(sl["user_id"].value_counts())

sl.loc[sl["user_id"] == "8378563200", ["date", "hour", "intensity", "calories"]].head(8)

Step-free effort hours: 18 · people: 3
user_id
8378563200    16
2873212765     1
6290855005     1
Name: count, dtype: int64


,date,hour,intensity,calories
15642,2016-04-12,5,165,669
15666,2016-04-13,5,165,669
15690,2016-04-14,5,150,616
15786,2016-04-18,5,165,669
15810,2016-04-19,5,165,669
15834,2016-04-20,5,165,669
15858,2016-04-21,5,165,669
15882,2016-04-22,5,165,669


In [118]:
hr_ids = profile.loc[profile["typical_resting_hr"].notna(), "user_id"]
hr_days = daily.dropna(subset=["minutes_hr_above_100", "mvpa_min"])
hr_days = hr_days[hr_days["user_id"].isin(hr_ids)]

within = (hr_days.groupby("user_id")
                        .apply(lambda g: g["mvpa_min"].rank().corr(g["minutes_hr_above_100"].rank()))
                 .sort_values())

print("People:", len(within), "· median within-person correlation:", round(within.median(), 2))
within.round(2)

People: 12 · median within-person correlation: 0.74


user_id
6117666160    0.36
2347167796    0.41
4388161847    0.52
7007744171    0.64
4558609924    0.66
8792009665    0.69
4020332650    0.80
6962181067    0.84
2022484408    0.86
5553957443    0.87
8877689391    0.93
5577150313    0.95
dtype: float64

In [119]:
print(nights.groupby("is_weekend_morning")["night_hours_asleep"].agg(["size", "median"]))

med = nights.pivot_table(index="user_id", columns="is_weekend_morning",
                         values="night_hours_asleep", aggfunc="median")
cnt = nights.pivot_table(index="user_id", columns="is_weekend_morning",
                         values="night_hours_asleep", aggfunc="size")
enough = (cnt[False] >= 3) & (cnt[True] >= 2)
med = med[enough]
med["diff_h"] = med[True] - med[False]

print("People compared:", len(med), "· sleep longer at weekends:", (med["diff_h"] > 0).sum())
med.round(2).sort_values("diff_h")

                    size  median
is_weekend_morning              
False                284   7.165
True                 100   7.725
People compared: 15 · sleep longer at weekends: 8


is_weekend_morning,False,True,diff_h
user_id,,,
6117666160,8.22,6.73,-1.49
8792009665,7.18,5.72,-1.47
4445114986,6.75,6.02,-0.73
3977333714,4.87,4.44,-0.42
5577150313,7.18,6.82,-0.36
6962181067,7.38,7.08,-0.30
2347167796,7.44,7.37,-0.08
4319703577,8.20,8.43,0.22
2026352035,8.50,8.75,0.26


In [120]:
naps = sleep[sleep["took_nap"]]
print("Sleep dates:", len(sleep), "· with a nap:", len(naps), f"({len(naps)/len(sleep):.0%})")
print("People who napped:", naps["user_id"].nunique(), "of", sleep["user_id"].nunique())
print("Typical nap:", naps["nap_minutes"].median(), "min")

print(sleep.groupby("is_weekend_morning")["took_nap"].mean().round(3))
naps["user_id"].value_counts()

Sleep dates: 410 · with a nap: 40 (10%)
People who napped: 13 of 24
Typical nap: 93.0 min
is_weekend_morning
False    0.064
True     0.188
Name: took_nap, dtype: float64


user_id
4388161847    6
4445114986    6
5553957443    6
3977333714    4
1644430081    3
6117666160    3
8378563200    3
1927972279    2
5577150313    2
7007744171    2
1503960366    1
4319703577    1
8053475328    1
Name: count, dtype: int64

In [121]:
cols = ["user_id", "typical_bedtime", "typical_wake_time", "bedtime_group",
        "bedtime_regularity", "typical_sleep_hours", "reliable_nights"]
bed_people = profile.dropna(subset=["bedtime_group"])[cols].sort_values("typical_sleep_hours")

print(bed_people["bedtime_group"].value_counts())
bed_people

bedtime_group
Typical    8
Late       4
Early      4
Name: count, dtype: int64


,user_id,typical_bedtime,typical_wake_time,bedtime_group,bedtime_regularity,typical_sleep_hours,reliable_nights
11,3977333714,12:52 AM,8:40 AM,Late,0.97,4.85,27
0,1503960366,2:23 AM,8:33 AM,Late,1.01,5.63,25
12,4020332650,11:28 PM,7:15 AM,Typical,1.18,6.42,7
16,4445114986,2:14 AM,10:03 AM,Late,1.42,6.61,26
18,4702921684,10:41 PM,6:03 AM,Typical,0.62,7.02,27
31,8792009665,2:19 AM,9:39 AM,Late,0.60,7.05,15
15,4388161847,10:07 PM,5:03 AM,Early,1.42,7.09,20
20,5577150313,10:09 PM,5:50 AM,Early,3.77,7.18,25
29,8378563200,9:32 PM,4:24 AM,Early,1.44,7.22,31
19,5553957443,11:05 PM,6:09 AM,Typical,1.04,7.22,31


In [122]:
x = (daily[daily["sleep_tonight_reliable_night"] == True]
          .dropna(subset=["mvpa_min", "sleep_tonight_night_hours_asleep"]).copy())
x["busy_day"] = x["mvpa_min"] > x.groupby("user_id")["mvpa_min"].transform("median")

res = (x.groupby(["user_id", "busy_day"])
         .agg(sleep_h=("sleep_tonight_night_hours_asleep", "median"),
              nights=("sleep_tonight_night_hours_asleep", "size"))
         .unstack())
res = res[(res["nights"][True] >= 3) & (res["nights"][False] >= 3)]
diff = (res["sleep_h"][True] - res["sleep_h"][False]).sort_values()

print("People compared:", len(diff), "· sleep longer after busier days:", (diff > 0).sum(),
      "· median difference:", round(diff.median() * 60), "min")
diff.round(2)

People compared: 14 · sleep longer after busier days: 10 · median difference: 14 min


user_id
4445114986   -1.48
8792009665   -1.33
5577150313   -0.23
4319703577   -0.05
6117666160    0.06
6962181067    0.15
7086361926    0.22
4702921684    0.25
2347167796    0.29
1503960366    0.37
5553957443    0.45
8378563200    0.60
3977333714    0.78
4388161847    1.25
dtype: float64

In [123]:
y = (daily[daily["sleep_last_night_reliable_night"] == True]
          .dropna(subset=["sleep_last_night_night_hours_asleep"]).copy())
y["short_night"] = (y["sleep_last_night_night_hours_asleep"]
                    < y.groupby("user_id")["sleep_last_night_night_hours_asleep"].transform("median"))

res2 = (y.groupby(["user_id", "short_night"])
          .agg(steps=("steps", "median"), days=("steps", "size"))
          .unstack())
res2 = res2[(res2["days"][True] >= 3) & (res2["days"][False] >= 3)]
diff2 = (res2["steps"][True] - res2["steps"][False]).sort_values()

print("People compared:", len(diff2), "· fewer steps after a short night:", (diff2 < 0).sum(),
      "· median difference:", round(diff2.median()), "steps")
diff2.round(0)

People compared: 16 · fewer steps after a short night: 5 · median difference: 917 steps


user_id
4020332650   -1180.0
2347167796    -609.0
4702921684    -409.0
6117666160    -204.0
7086361926    -138.0
4388161847     134.0
6962181067     157.0
5577150313     774.0
2026352035    1060.0
4319703577    1271.0
8378563200    1353.0
3977333714    1528.0
4445114986    1728.0
1503960366    2278.0
8792009665    3224.0
5553957443    7190.0
dtype: float64

In [124]:
sp = profile.dropna(subset=["typical_sleep_hours", "activity_segment"])

print(sp.groupby("activity_segment")
        .agg(people=("user_id", "size"),
             sleep_h=("typical_sleep_hours", "median"),
             efficiency=("typical_sleep_efficiency", "median"))
        .reindex(seg_order).round(2))

print("People-level correlation, steps vs sleep:",
      round(sp["typical_steps"].corr(sp["typical_sleep_hours"], method="spearman"), 2))

profile.groupby("activity_segment")["reliable_nights"].median().reindex(seg_order)

                  people  sleep_h  efficiency
activity_segment                             
Sedentary              2     6.83        0.95
Low active             2     7.52        0.93
Somewhat active        5     7.22        0.95
Active                 7     7.22        0.94
People-level correlation, steps vs sleep: -0.09


activity_segment
Sedentary           4.0
Low active          0.0
Somewhat active     9.0
Active             20.0
Name: reliable_nights, dtype: float64

### Engagement & retention (T6)

In [125]:
status_order  = ["worn", "partial", "not_worn", "last_day", "stopped"]
status_colors = {"worn": "#2a9d8f", "partial": "#e9c46a", "not_worn": "#e76f51",
                 "last_day": "#a8b5c4", "stopped": "#d9d9d9"}

user_order = (wear[wear["wear_status"] == "worn"].groupby("user_id").size()
                  .reindex(wear["user_id"].unique(), fill_value=0)
                  .sort_values(ascending=False).index.tolist())

fig = px.scatter(wear, x="date", y="user_id", color="wear_status",
                 category_orders={"wear_status": status_order, "user_id": user_order},
                 color_discrete_map=status_colors,
                 title="Most People Wear It Almost Every Day; Gaps Come From a Few Users and Drop-Outs"
                       "<br><sup>One square per person per day · 33 people × 31 days · sorted by days worn</sup>",
                 labels={"date": "", "user_id": "", "wear_status": "Status"})
fig.update_traces(marker=dict(symbol="square", size=13),
                  hovertemplate="%{y}<br>%{x|%a %d %b}: %{fullData.name}<extra></extra>")
fig.update_yaxes(type="category")
fig.update_layout(height=850)
fig.show()

In [126]:
in_study = wear[~wear["wear_status"].isin(["stopped", "last_day"])]
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

by_day = (in_study.groupby("day_of_week")["wear_status"]
                  .value_counts(normalize=True)
                  .unstack()
                  .reindex(order)[["worn", "partial", "not_worn"]]
                  .round(3))
by_day

wear_status,worn,partial,not_worn
day_of_week,,,
Monday,0.815,0.092,0.092
Tuesday,0.887,0.027,0.087
Wednesday,0.844,0.068,0.088
Thursday,0.833,0.087,0.079
Friday,0.855,0.089,0.056
Saturday,0.826,0.091,0.083
Sunday,0.767,0.125,0.108


In [127]:
tier_order = ["Barely using", "Irregular", "Consistent"]
ct = (pd.crosstab(seg["activity_segment"], seg["engagement_tier"])
        .reindex(index=seg_order, columns=tier_order, fill_value=0))

fig = px.imshow(ct, text_auto=True, color_continuous_scale="Oranges", aspect="auto",
                title="7 of 10 Low-Steppers Use the Device Irregularly, vs 4 of 21 Active Users"
                      "<br><sup>People per activity segment × engagement tier · 31 people with 7+ clean days</sup>",
                labels={"x": "Engagement tier", "y": "", "color": "People"})
fig.show()
ct

engagement_tier,Barely using,Irregular,Consistent
activity_segment,,,
Sedentary,1,2,2
Low active,1,3,1
Somewhat active,1,1,8
Active,1,1,9


In [128]:
tr = profile.dropna(subset=["activity_trend"])
long_tr = tr.melt(id_vars=["user_id", "activity_trend", "engagement_tier"],
                  value_vars=["steps_1st_half", "steps_2nd_half"],
                  var_name="half", value_name="steps")
long_tr["half"] = long_tr["half"].map({"steps_1st_half": "12–26 Apr",
                                       "steps_2nd_half": "27 Apr – 11 May"})

fig = px.line(long_tr, x="half", y="steps", color="activity_trend", line_group="user_id",
              markers=True, hover_data=["user_id", "engagement_tier"],
              category_orders={"activity_trend": ["Falling", "Stable", "Rising"]},
              color_discrete_map={"Falling": "#e76f51", "Stable": "#b0b0b0", "Rising": "#2a9d8f"},
              title="12 People Slowed Down in the Second Half of the Month; 7 Sped Up"
                    "<br><sup>Median daily steps per person, first vs second half · within ±10% = stable · 31 people</sup>",
              labels={"half": "", "steps": "Median daily steps", "activity_trend": "Trend"})
fig.update_yaxes(rangemode="tozero")
fig.update_layout(height=600)
fig.show()

pd.crosstab(tr["activity_trend"], tr["engagement_tier"])

engagement_tier,Barely using,Consistent,Irregular
activity_trend,,,
Falling,2,6,4
Rising,0,7,0
Stable,2,7,3


In [129]:
funnel = pd.DataFrame({
    "stage": ["All users", "Tried sleep tracking",
              "Regular (7+ reliable nights)", "Consistent (20+ reliable nights)"],
    "people": [len(profile), int(profile["tracks_sleep"].sum()),
               int((profile["reliable_nights"] >= 7).sum()),
               int((profile["reliable_nights"] >= 20).sum())],
})
funnel["share"] = (funnel["people"] / len(profile)).map("{:.0%}".format)

fig = px.funnel(funnel, x="people", y="stage", text="share",
                title="Sleep Tracking: 24 People Tried It, but Only 12 Kept It Up"
                      "<br><sup>People at each stage of sleep-feature adoption · 33 people</sup>",
                labels={"stage": "", "people": "People"})
fig.update_traces(marker_color="#FC4C02")
fig.show()

print("Weight: logged at least once:", int(profile["logs_weight"].sum()),
      "· habit (10+ logs):", int(profile["weight_habit"].fillna(False).astype(bool).sum()))
print("Workouts: logged manually:", int(profile["logs_workouts"].sum()))
funnel

Weight: logged at least once: 8 · habit (10+ logs): 2
Workouts: logged manually: 3


,stage,people,share
0,All users,33,100%
1,Tried sleep tracking,24,73%
2,Regular (7+ reliable nights),16,48%
3,Consistent (20+ reliable nights),12,36%


In [130]:
hs = hourly.merge(seg[["user_id", "activity_segment"]], on="user_id")
seg_hour = hs.groupby(["activity_segment", "hour"])["steps"].mean().reset_index()

fig = px.line(seg_hour, x="hour", y="steps", color="activity_segment",
              facet_col="activity_segment", facet_col_wrap=2,
              category_orders={"activity_segment": seg_order}, color_discrete_map=seg_colors,
              title="Active People Are Set Apart by Their After-Work Hours (5–7 PM)"
                    "<br><sup>Average steps per hour of day, by activity segment · 31 people · shaded = 5–7 PM</sup>",
              labels={"hour": "Hour of day", "steps": "Avg steps", "activity_segment": "Segment"})
fig.add_vrect(x0=17, x1=19, fillcolor="orange", opacity=0.12, line_width=0)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_traces(line_width=3, hovertemplate="%{x}:00 · %{y:,.0f} steps<extra></extra>")
fig.update_layout(height=600, showlegend=False)
fig.show()

### Layer 4 · Key findings
- **Segments:** Active 11 · Somewhat active 10 · Low active 5 · Sedentary 5 (31 people). **7 people are within 1,000 steps of the next level** (two within 100).
- **Steps vs guideline agree:** all Active meet 150 MVPA min/week, none of the bottom 10 do; Somewhat active split 7/3. No hidden step-free exercisers.
- **Pace, not time:** apart from Sedentary, all segments are active ~280–290 min/day; the difference is effort (Active: 21% of active time is moderate-vigorous vs ~3% for the bottom two). Distance confirms it (very-active km 0.26 → 3.21).
- **Calories don't rank activity** (body size drives them) → not used for comparisons.
- **Routines:** more regular routines go with more steps (6.6k Irregular → 9.8k Very regular); CV can be inflated by one huge day.
- **Effort data is trustworthy:** band MVPA agrees with heart rate for all 12 HR users (median within-person correlation 0.74).
- **Step-free workouts:** almost none — mostly one person's manually logged 5 AM sessions; only 3 people ever log workouts.
- **Weekend sleep:** pooled +34 min, but per person split 8 longer / 7 shorter — a few big catch-up sleepers drive it. Naps 3× more common on weekends; 13 of 24 sleep trackers napped.
- **Bedtime:** 3 of 4 short sleepers go to bed late with a fixed wake time; night owls who sleep in are fine.
- **Activity → sleep:** busier days → ~15 min more sleep for 10 of 14 people — a hint, not proof. **Sleep → activity:** after short nights people walk *more* (early, busy days cut sleep).
- **Segments sleep alike** (correlation ≈ 0) but active users **track** sleep far more (20 vs 0–4 reliable nights).
- **Engagement:** activity and wear go together (wear share 52% Sedentary → 94% Active); 5 users cause 64% of missed/partial days; **Sunday worst (77% worn), Tuesday best (89%)**; some drop-outs stop with no warning.
- **Persona grid:** 17 of 31 are active + consistent; 7 of 10 low-steppers use the device irregularly; 1 "Barely using" in every segment.
- **Trend per person:** flat on average, but **12 slowed by 10%+ and 7 sped up**; all risers are consistent wearers; 6 people are falling + irregular → churn risk.
- **Feature funnel:** sleep 33 → 24 tried → 16 regular → 12 consistent (biggest relative loss: tried → regular, often failed recordings); weight 8 → 2 habit; workouts 3.
- **Timing by segment:** Active people build their lead **after work (5–7 PM, ~3× Low active)**; daytime activity is similar across segments.

## Layer 5 · Marketing synthesis
### Personas (Q29)

In [131]:
def persona(row):
    if row["engagement_tier"] == "Barely using" or pd.isna(row["activity_segment"]):
        return "Fading Users"
    higher = row["activity_segment"] in ("Somewhat active", "Active")
    if row["engagement_tier"] == "Consistent":
        return "Committed Movers" if higher else "Daily Strollers"
    return "On-and-Off Movers" if higher else "Slipping Starters"

profile["persona"] = profile.apply(persona, axis=1)

persona_order = ["Committed Movers", "On-and-Off Movers", "Daily Strollers",
                 "Slipping Starters", "Fading Users"]
profile["persona"].value_counts().reindex(persona_order)

persona
Committed Movers     17
On-and-Off Movers     2
Daily Strollers       3
Slipping Starters     5
Fading Users          6
Name: count, dtype: int64

In [132]:
persona_profile = (profile.groupby("persona")
    .agg(people=("user_id", "size"),
         typical_steps=("typical_steps", "median"),
         weekly_mvpa=("weekly_mvpa_min", "median"),
         wear_share=("clean_day_share", "median"),
         meets_guideline=("meets_activity_guideline", lambda s: (s == True).mean()),
         tracks_sleep=("tracks_sleep", "mean"),
         features_used=("features_used", "mean"),
         falling=("activity_trend", lambda s: (s == "Falling").mean()),
         stopped_early=("stopped_early", "mean"))
    .reindex(persona_order)
    .round(2))
persona_profile

,people,typical_steps,weekly_mvpa,wear_share,meets_guideline,tracks_sleep,features_used,falling,stopped_early
persona,,,,,,,,,
Committed Movers,17,10243.0,334.0,0.90,0.88,0.76,1.29,0.29,0.00
On-and-Off Movers,2,11453.5,200.5,0.69,0.50,1.00,1.50,0.50,0.00
Daily Strollers,3,4974.0,60.0,0.94,0.00,0.67,0.67,0.33,0.00
Slipping Starters,5,5205.0,46.0,0.52,0.00,0.80,0.80,0.60,0.00
Fading Users,6,6540.0,172.5,0.36,0.33,0.50,0.67,0.33,0.67


### Persona sheet
| Persona | People | Who they are | What they need | Best timing / feature |
|---|---|---|---|---|
| **Committed Movers** | 17 | ~10.2k steps, 334 MVPA min/wk, 90% wear, 76% track sleep; 5 slowing down | Keep them: streaks, milestones, challenges, routes; "below your usual" nudges | Evening content 5–7 PM; personalised by workout time |
| **On-and-Off Movers** | 2 | Very active when worn (11.5k) but 69% wear; both vanished in final week | Wear & sync consistency | Charge/sync reminders; weekend wear nudge |
| **Daily Strollers** | 3 | 94% wear but ~5k steps, 60 MVPA min/wk; none meet guideline | Pace, not time: next-level goals, "15 brisk minutes" | Daily goal nudges; ~5 PM evening walk |
| **Slipping Starters** | 5 | ~5.2k steps, 52% wear, 60% slowing; 80% track sleep | Re-engage before they fade; small wins; sleep as the hook | "We missed you" after 1st missed day; Sun/Mon reminders |
| **Fading Users** | 6 | 36% wear; 4 of 6 stopped early | Win-back; stronger first 2 weeks | Onboarding in days 1–14; exit feedback |

*Small groups (2–6 people) → treat persona traits as directional, not precise.*

### Top 5 recommendations (Q30)

| # | Recommendation | Evidence from the EDA | Who | Confidence |
|---|---|---|---|---|
| 1 | **Retention first — protect the wearing habit.** Streak rewards, a "we missed you" nudge after the *first* missed day, Saturday-evening charge + Sunday/Monday wear reminders. | Drop-outs lost more days (83) than non-wear (77); for people who keep wearing, activity doesn't fade; **all 7 people who improved were consistent wearers**; Sunday 77% worn vs Tuesday 89%; 5 users cause 64% of gaps. | Slipping Starters, On-and-Off, Fading | Medium-high (33 people, 1,023 person-days) |
| 2 | **Coach pace, not time — "15 brisk minutes a day".** Turn part of the walking people already do into brisk walking. | Low/Somewhat/Active are all active ~280–290 min/day; the gap is effort (very-active km 0.26 vs 3.21). Bottom 10 do ~50 MVPA min/week vs 150 guideline. Effort data confirmed by heart rate (r = 0.74). | Daily Strollers, Slipping Starters | High (two measures + HR check) |
| 3 | **Smarter goals — next-level targets and personal baselines, not one 10k for all.** | ~50% more days end just past 10k than just before → goals change behaviour. 7 people are within 1,000 steps of the next level; bottom segments almost never reach 10k; 12 people slowed 10%+ — invisible to a fixed goal. | All (esp. Daily Strollers, slowing Committed Movers) | Medium |
| 4 | **Time every message to the person's rhythm.** Weekdays ~5 PM, weekends ~10–11 AM, a 3 PM "move" prompt, wind-down timed from wake-up time; no early-weekend or late-night pushes. | Weekday peak 5–7 PM; weekend peak 10 AM–2 PM (Saturday 1 PM busiest); Active users build their lead after work (~3× at 6 PM); workout preference differs (Morning 13 · Afternoon 9 · Evening 8). | All | Medium-high (17,987 hours) |
| 5 | **Make tracking effortless — automatic beats manual.** Fix the first week of sleep tracking (setup tips, 7-night challenge); smart-scale sync; auto-detected and "repeat last" workouts. | Adoption: sleep 73% → weight 24% → workouts 9%. Sleep keeps half its triers, weight a quarter. A third of sleep triers never got a week of usable nights (often failed recordings). One user typed the same workout 16 times. | Daily Strollers, Slipping Starters, Committed Movers | Medium (small n for weight/workouts) |

**What we deliberately do *not* claim:** "move more, sleep better" (only a +15 min hint), any health or BMI conclusions (n = 8, self-selected), calories as an activity measure, or anything causal — this is 33 people over one month in 2016.